# Train the Transformer on Colab (T13)

1. **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
2. The repo is cloned into your Google Drive, so checkpoints and results survive a disconnect.
   If Colab disconnects, run all cells again: finished runs are skipped.

In [1]:
!nvidia-smi

Sun Nov  8 21:13:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Clone the repo into Drive (or update it if it is already there), install it and build the token ids.

In [3]:
REPO = "/content/drive/MyDrive/language-models-from-scratch"
!test -d {REPO} || git clone https://github.com/mohamadalichamseddine/language-models-from-scratch.git {REPO}
%cd {REPO}
!git pull
!pip install -q -r requirements.txt
!python scripts/prepare_data.py

Cloning into '/content/drive/MyDrive/language-models-from-scratch'...
remote: Enumerating objects: 301, done.
remote: Counting objects: 100% (301/301), done.
remote: Compressing objects: 100% (161/161), done.
remote: Total 301 (delta 149), reused 243 (delta 109), pack-reused 0 (from 0)
Receiving objects: 100% (301/301), 4.44 MiB | 11.91 MiB/s, done.
Resolving deltas: 100% (149/149), done.
Updating files: 100% (106/106), done.
/content/drive/MyDrive/language-models-from-scratch
Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for language-models-from-scratch (pyproject.toml) ... done
vocab: 33,278 tokens -> /content/drive/MyDrive/language-models-from-scratch/data/processed/vocab.json
train: 2,088,628 tokens, 54,625 <unk>, checksum d5163de236cc
valid: 217,646 tokens, 11,718 <unk>, checksum 9b

Run the tests, then a 30-step smoke run to check that training works on the GPU.
Its output shows how many seconds 10 steps take (a full run is at most 4,000 steps).

In [4]:
!python -m pytest -q
!python scripts/train_transformer.py --run smoke --max-steps 30 --eval-every 10 --eval-tokens 20000 --results-dir /content/smoke --checkpoint-dir /content/smoke

........................................................................ [ 86%]
...........                                                              [100%]
83 passed in 17.07s
Training 11,711,488 parameters on cuda, context 128
step     0 | train   -    | val 10.4467 | 0s
step    10 | train 10.3741 | val 10.1332 | 2s
step    20 | train 10.0463 | val 9.8065 | 4s
step    30 | train 9.7759 | val 9.5731 | 5s
Best val loss 9.5731 at step 30 (0.1 min on Tesla T4)


## The sweep

Trains and scores 9 runs at context 128: learning rate 3e-4, 6e-4, 1e-3 × dropout 0.1, 0.2, 0.3.
It writes `results/transformer/sweep/`, the frozen best settings in `configs/transformer.json`,
and the comparison table `results/comparison.md`.

In [5]:
!python scripts/sweep_transformer.py


=== Training transformer/sweep/lr0.0003_drop0.1 ===
Training 11,711,488 parameters on cuda, context 128
step     0 | train   -    | val 10.4577 | 2s
step   100 | train 9.4047 | val 7.9212 | 16s
step   200 | train 7.0550 | val 6.3407 | 31s
step   300 | train 6.2942 | val 5.9351 | 46s
step   400 | train 5.9713 | val 5.7366 | 63s
step   500 | train 5.7640 | val 5.5950 | 81s
step   600 | train 5.6193 | val 5.4814 | 98s
step   700 | train 5.4929 | val 5.3934 | 115s
step   800 | train 5.3911 | val 5.3258 | 131s
step   900 | train 5.2963 | val 5.2636 | 148s
step  1000 | train 5.2107 | val 5.2050 | 165s
step  1100 | train 5.1255 | val 5.1633 | 182s
step  1200 | train 5.0670 | val 5.1193 | 198s
step  1300 | train 5.0000 | val 5.0840 | 215s
step  1400 | train 4.9422 | val 5.0458 | 232s
step  1500 | train 4.8895 | val 5.0210 | 249s
step  1600 | train 4.8375 | val 4.9865 | 266s
step  1700 | train 4.8015 | val 4.9641 | 283s
step  1800 | train 4.7487 | val 4.9485 | 299s
step  1900 | train 4.7099 | 

In [6]:
!cat results/transformer/sweep/sweep.md results/comparison.md configs/transformer.json

# Transformer sweep at context 128

Learning rate x dropout; every other setting is the default of scripts/train_transformer.py.
Validation perplexity comes from sliding-window scoring. The best run is in bold.

| Learning rate | Dropout | Val perplexity | Top-1 | Best step | Steps trained | Minutes |
|---|---|---|---|---|---|---|
| 0.0003 | 0.1 | 114.38 | 25.22% | 4000 | 4000 | 11.2 |
| 0.0003 | 0.2 | 122.44 | 24.33% | 4000 | 4000 | 11.3 |
| 0.0003 | 0.3 | 136.15 | 23.26% | 3900 | 4000 | 11.3 |
| 0.0006 | 0.1 | **107.24** | 25.48% | 3300 | 3800 | 10.7 |
| 0.0006 | 0.2 | 107.70 | 25.36% | 4000 | 4000 | 11.2 |
| 0.0006 | 0.3 | 117.91 | 24.25% | 4000 | 4000 | 11.2 |
| 0.001 | 0.1 | 113.25 | 25.41% | 2400 | 2900 | 8.2 |
| 0.001 | 0.2 | 107.64 | 25.70% | 4000 | 4000 | 11.3 |
| 0.001 | 0.3 | 114.06 | 24.68% | 4000 | 4000 | 11.3 |

Trained on: Tesla T4.
Metrics on the valid split (217,646 tokens). N-grams use their best k.

| Run | CE (nats) | CE (bits) | Perplexity | Top-1 | Top-5 | Top-10 

## Download the results

The zip holds the run folders, the tables, the frozen config and the best run's checkpoint,
which later tickets need (T14, T17). Unzip it in the repo root on your laptop and commit
everything in it. Left out: the `.npz` prediction files (`scripts/score_transformer.py`
rebuilds them) and the other runs' checkpoints, which stay in Drive.

In [7]:
import json
best_run = json.load(open("results/transformer/sweep/sweep.json"))["best_run"]
!zip -r /content/t13_results.zip results/transformer results/comparison.md results/comparison.csv configs/transformer.json checkpoints/{best_run}/best.pt -x "*.npz"
from google.colab import files
files.download("/content/t13_results.zip")

  adding: results/transformer/ (stored 0%)
  adding: results/transformer/sweep/ (stored 0%)
  adding: results/transformer/sweep/lr0.0003_drop0.1/ (stored 0%)
  adding: results/transformer/sweep/lr0.0003_drop0.1/config.json (deflated 48%)
  adding: results/transformer/sweep/lr0.0003_drop0.1/curves.csv (deflated 52%)
  adding: results/transformer/sweep/lr0.0003_drop0.1/training.json (deflated 31%)
  adding: results/transformer/sweep/lr0.0003_drop0.1/metrics.json (deflated 41%)
  adding: results/transformer/sweep/lr0.0003_drop0.2/ (stored 0%)
  adding: results/transformer/sweep/lr0.0003_drop0.2/config.json (deflated 48%)
  adding: results/transformer/sweep/lr0.0003_drop0.2/curves.csv (deflated 51%)
  adding: results/transformer/sweep/lr0.0003_drop0.2/training.json (deflated 31%)
  adding: results/transformer/sweep/lr0.0003_drop0.2/metrics.json (deflated 41%)
  adding: results/transformer/sweep/lr0.0003_drop0.3/ (stored 0%)
  adding: results/transformer/sweep/lr0.0003_drop0.3/config.json (

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>